In [1]:
import os

In [2]:
%pwd

'c:\\Users\\AJAY\\Documents\\ML Projects\\TextSummarizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\AJAY\\Documents\\ML Projects\\TextSummarizer'

### Basic Configuration

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class ModelTrainingConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs : int
    warmup_steps : int
    per_device_train_batch_size : int
    weight_decay : float
    logging_steps : int
    eval_strategy: str
    eval_steps : int
    save_steps : float
    gradient_accumulation_steps : int

In [6]:
from src.TextSummarizer.constants import *
import unittest

# Compatibility fix for libraries using the removed Python 2/older Python API
if not hasattr(unittest.TestCase, "assertRaisesRegexp"):
    unittest.TestCase.assertRaisesRegexp = unittest.TestCase.assertRaisesRegex

from src.TextSummarizer.utils.common import read_yaml, create_directories

### Configuration update

In [7]:
class ConfigurationManager:
    def __init__(self,
                 config_path = CONFIG_FILE_PATH,
                 params_path =  PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainingConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_traniner_config = ModelTrainingConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            model_ckpt =  config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps  = params.logging_steps,
            eval_strategy = params.eval_strategy,
            eval_steps  = params.eval_steps,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
        )
        return model_traniner_config
        

### Components

In [8]:
import os
from src.TextSummarizer.logging import logger


from transformers import AutoModelForSeq2SeqLM,AutoTokenizer
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
import torch

from datasets import load_from_disk


c:\Users\AJAY\Documents\ML Projects\TextSummarizer\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class ModelTrainer:
    def __init__(self,config: ModelTrainingConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt) 
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        ## Loading the data
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
                    output_dir=self.config.root_dir,
                    num_train_epochs=1,
                    warmup_steps=500,
                    per_device_train_batch_size=1,
                    per_device_eval_batch_size=1,
                    weight_decay=0.01,
                    logging_steps=10,
                    eval_strategy='steps',
                    eval_steps=500,
                    save_steps=1e6,
                    gradient_accumulation_steps=16,
                    dataloader_pin_memory=False
                )

        trainer = Trainer(model=model_pegasus, args=trainer_args,
                  data_collator=seq2seq_data_collator,
                  train_dataset=dataset_samsum_pt["test"],
                  eval_dataset=dataset_samsum_pt["validation"])


        trainer.train()

        ## Save model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir,"pegasus-samsum-model"))

        ## Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir,"tokenizer"))


In [12]:
# !pip install --upgrade accelerate
# !pip uninstall -y transformers accelerate
# !pip install transformers accelerate

In [13]:
config = ConfigurationManager()
model_trainer_config = config.get_model_trainer_config()
model_trainer = ModelTrainer(config=model_trainer_config)
model_trainer.train()

[2026-09-03 07:56:04,797: INFO: common]: yaml file: config\config.yaml loaded successfully]
[2026-09-03 07:56:04,811: INFO: common]: yaml file: params.yaml loaded successfully]
[2026-09-03 07:56:04,813: INFO: common]: created directory at: artifacts]
[2026-09-03 07:56:04,815: INFO: common]: created directory at: artifacts/model_trainer]
[2026-09-03 07:56:05,642: INFO: _client]: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"]
[2026-09-03 07:56:05,874: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-09-03 07:56:05,887: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-09-03 07:56:06,205: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.

[2026-09-03 07:56:12,377: WARNING: _http]: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-09-03 07:56:13,002: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/refs%2Fpr%2F12/model.safetensors "HTTP/1.1 302 Found"]


Loading weights: 100%|██████████| 680/680 [00:00<00:00, 4528.20it/s]
[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[2026-09-03 07:56:38,449: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-09-03 07:56:38,653: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/generation_config.json "HTTP/1.1 200 OK"]


c:\Users\AJAY\Documents\ML Projects\TextSummarizer\env\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


: 

In [11]:
import torch
import transformers

print('torch.cuda.is_available():', torch.cuda.is_available())
print('torch.cuda.device_count():', torch.cuda.device_count())
print('transformers version:', getattr(transformers, '__version__', 'unknown'))


torch.cuda.is_available(): False
torch.cuda.device_count(): 0
transformers version: 5.16.1
